In [ ]:
# Config
from pathlib import Path
import os

# Defaults - change as needed
OUTPUTS_ROOT = Path('training_gradient_evaluator_single_loss/outputs')
RESULTS_CSV_NAME = 'single_example_results.csv'
SEED = 1337
TEST_SIZE = 0.10
VAL_SIZE = 0.05  # of the total dataset (will be normalized after test split)
TARGET_COLUMN = 'universal_difficulty_rank'
FEATURE_COLUMNS = [
    'total_steps_to_epsilon',
    'total_loss_sum',
    'final_loss',
    'weight_distance',
    'softmax_wasserstein',
    'grad_mass_wasserstein',
    'global_linear_cka',
]
SAVE_MODELS = True
ARTIFACTS_DIRNAME = 'rank_predictor_artifacts'



In [ ]:
# Imports
import json
import math
import random
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_hist_gradient_boosting  # noqa: F401
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.multioutput import RegressorChain
from sklearn.base import BaseEstimator, RegressorMixin

import joblib

random.seed(SEED)
np.random.seed(SEED)



In [ ]:
# Utilities

def find_model_result_csvs(root: Path, csv_name: str) -> dict:
	"""Return mapping: model_dir_path -> csv_path."""
	root = Path(root)
	mapping = {}
	for model_dir in sorted(p for p in root.iterdir() if p.is_dir()):
		csv_path = model_dir / csv_name
		if csv_path.exists():
			mapping[str(model_dir)] = str(csv_path)
	return mapping


def load_dataset(csv_path: str, feature_cols: list[str], target_col: str) -> pd.DataFrame:
	df = pd.read_csv(csv_path)
	# Ensure required columns exist; missing feature columns will be added as NaN
	for c in feature_cols + [target_col, 'path', 'example_index']:
		if c not in df.columns:
			df[c] = np.nan
	return df


def split_train_val_test(df: pd.DataFrame, test_size: float, val_size: float, seed: int):
	# First split off test
	train_val, test = train_test_split(df, test_size=test_size, random_state=seed, shuffle=True)
	# Val proportion relative to remaining
	val_rel = val_size / max(1e-9, (1.0 - test_size))
	train, val = train_test_split(train_val, test_size=val_rel, random_state=seed, shuffle=True)
	return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


def build_regressors(seed: int):
	models = {
		"hgb": HistGradientBoostingRegressor(random_state=seed, max_depth=None, learning_rate=0.05, max_iter=500),
		"rf": RandomForestRegressor(random_state=seed, n_estimators=400, max_depth=None, n_jobs=-1, oob_score=False),
		"ridge": Ridge(random_state=seed),
	}
	return models


def build_pipeline(estimator, numeric_features: list[str]) -> Pipeline:
	preprocess = ColumnTransformer(
		transformers=[
			("num", Pipeline(steps=[
				("imputer", SimpleImputer(strategy="median")),
				("scaler", StandardScaler()),
			]), numeric_features),
		], remainder='drop'
	)
	pipe = Pipeline(steps=[
		("preprocess", preprocess),
		("regressor", estimator),
	])
	return pipe


def evaluate_model(model: Pipeline, X: pd.DataFrame, y: np.ndarray) -> dict:
	pred = model.predict(X)
	mae = float(mean_absolute_error(y, pred))
	r2 = float(r2_score(y, pred))
	return {"mae": mae, "r2": r2}


def fit_ensemble(train_df: pd.DataFrame, val_df: pd.DataFrame, feature_cols: list[str], target_col: str, seed: int):
	models = build_regressors(seed)
	metrics = {}
	best_name = None
	best_score = float('inf')
	best_model = None
	for name, est in models.items():
		pipe = build_pipeline(est, feature_cols)
		pipe.fit(train_df[feature_cols], train_df[target_col].values)
		m = evaluate_model(pipe, val_df[feature_cols], val_df[target_col].values)
		metrics[name] = m
		if m["mae"] < best_score:
			best_score = m["mae"]
			best_name = name
			best_model = pipe
	return best_name, best_model, metrics


def save_artifacts(model_dir: Path, model_name: str, best_name: str, model: Pipeline, metrics: dict, feature_cols: list[str]) -> None:
	art_dir = Path(model_dir) / ARTIFACTS_DIRNAME
	art_dir.mkdir(parents=True, exist_ok=True)
	joblib.dump(model, art_dir / f"best_model_{best_name}.joblib")
	with open(art_dir / "metrics.json", 'w', encoding='utf-8') as f:
		json.dump(metrics, f, indent=2)
	with open(art_dir / "features.json", 'w', encoding='utf-8') as f:
		json.dump({"feature_columns": feature_cols, "target": TARGET_COLUMN}, f, indent=2)



In [ ]:
# Main routine per model

def train_and_evaluate_for_csv(model_dir: str, csv_path: str,
                              feature_cols=FEATURE_COLUMNS,
                              target_col=TARGET_COLUMN,
                              seed=SEED,
                              test_size=TEST_SIZE,
                              val_size=VAL_SIZE,
                              save_models=SAVE_MODELS):
	print(f"Processing: {model_dir}")
	df = load_dataset(csv_path, feature_cols, target_col)
	# Drop rows without target or entirely missing features
	df = df.dropna(subset=[target_col])
	if df.empty:
		print("No data.")
		return None
	# Prepare splits
	train_df, val_df, test_df = split_train_val_test(df, test_size, val_size, seed)
	best_name, best_model, metrics = fit_ensemble(train_df, val_df, feature_cols, target_col, seed)
	# Evaluate on test set (holdout)
	test_metrics = evaluate_model(best_model, test_df[feature_cols], test_df[target_col].values)
	metrics['test'] = test_metrics
	print("Validation metrics:", metrics.get(best_name, {}))
	print("Test metrics:", test_metrics)
	if save_models and best_model is not None:
		save_artifacts(Path(model_dir), Path(model_dir).name, best_name, best_model, metrics, feature_cols)
	return {"model_dir": model_dir, "best_model": best_name, "metrics": metrics}


# Discover and process all model folders
model_csvs = find_model_result_csvs(OUTPUTS_ROOT, RESULTS_CSV_NAME)
results = []
for mdir, csvp in model_csvs.items():
	res = train_and_evaluate_for_csv(mdir, csvp)
	results.append(res)

results
